# MindTrack: Depression Score-Group Prediction Pipeline

This notebook guides a reviewer through the project from the source questionnaires to exploratory analysis, data preparation, PCA, and model evaluation. The single target is the dataset-defined group with Beck score **11 or higher**. This is a research score group, **not a clinical diagnosis**.

The selected phone-dependence input is the total recalculated from its 22 questionnaire answers. PCA is included to explore predictor structure and as an optional modeling comparison. PCA does **not** rank features by their effect on the target.

### Before running the notebook

Open this notebook with the Python environment you use for the project. From the project root, install the required packages if they are not already available:

```bash
python -m pip install -r analysis/eda/requirements-eda.txt
python -m pip install -r analysis/modeling/requirements-modeling.txt
```

Run the cells from top to bottom. The notebook calls the reproducible preparation, PCA, and modeling scripts and displays their outputs.

In [ ]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = Path.cwd()
if not (ROOT / 'analysis').exists():
    ROOT = ROOT.parent
assert (ROOT / 'analysis').exists(), 'Open/run this notebook from the project folder.'
print('Project folder:', ROOT)

## 1. Study question and scope

**Question:** How well can background characteristics and mobile-phone-use information predict whether a student's Beck score is in the dataset's 11+ score group?

The project is observational. A prediction or association does not show that phone use causes a higher Beck score. The dataset's codebook labels scores 0–10 as Normal and 11–16 as Mild mood disturbance; therefore the binary outcome combines all score bands beginning at 11. We retain the careful label “higher Beck-score group” and recommend confirming that scoring guide with the lecturer.

## 2. Source files and record alignment

The source folder contains separate sociodemographic/mobile-use, phone-dependence, and Beck workbooks. The check below confirms that the coded student records have unique IDs and align across the three files. It displays only structural counts, not personal row details.

In [ ]:
raw = ROOT / 'Digital Sedentary Behavior, Mobile Device Dependen'
sources = [
    ('Sociodemographic and mobile use', raw / '2. Dataset_Sociodemo.xlsx', 'SocioDemo_data'),
    ('Phone dependence', raw / '3. Dataset_MobileDependence.xlsx', 'TMD_data'),
    ('Beck scores', raw / '5. Dataset_Depression.xlsx', 'Depression_data'),
]
frames = {label: pd.read_excel(path, sheet_name=sheet) for label, path, sheet in sources}
inventory = pd.DataFrame([{'source': label, 'records': len(frame), 'columns': len(frame.columns), 'unique_codes': frame['Code'].nunique(), 'missing_codes': int(frame['Code'].isna().sum())} for label, frame in frames.items()])
display(inventory)
codes = [frame['Code'] for frame in frames.values()]
assert all(code.is_unique for code in codes), 'Duplicate/missing student codes need investigation.'
assert codes[0].equals(codes[1]) and codes[0].equals(codes[2]), 'Student records do not align across workbooks.'
print(f'All {len(codes[0]):,} records align across the three source files.')

## 3. Exploratory data analysis (EDA)

The EDA describes the Beck scores, phone-use patterns, and candidate predictors before modeling. The charts are saved in `analysis/eda/figures/presentation/`; the full findings and quality review are in `analysis/eda/preliminary_eda_findings.md` and `analysis/eda/tables/`.

Key observations: 1,220 records; Beck scores cluster near zero; 218 (17.9%) meet the dataset-defined 11+ grouping; the recomputed dependence total has a weak positive rank association with Beck total; one supplied dependence total disagreed with its item sum; the daily-use 1–3 hour category has only one response. These are descriptive findings, not causal or diagnostic conclusions.

### Beck score distribution and group definition

![Distribution of Beck total scores](../analysis/eda/figures/presentation/01_beck_score_distribution.png)

![Beck score bands from the project codebook](../analysis/eda/figures/presentation/02_beck_score_bands.png)

### Phone-use patterns and Beck scores

![Phone-dependence score and Beck total; see the quality note about one supplied total](../analysis/eda/figures/presentation/03_phone_dependence_vs_beck.png)

![Beck score by phone-dependence band](../analysis/eda/figures/presentation/04_beck_score_by_dependence_band.png)

![Higher-score-group rate by phone-related arguments](../analysis/eda/figures/presentation/05_beck_flag_by_phone_arguments.png)

![Daily phone-use category distribution](../analysis/eda/figures/presentation/06_average_daily_use_distribution.png)

### Other participant and response patterns

![Age distribution](../analysis/eda/figures/presentation/07_age_distribution.png)

![Higher-score-group rate by department](../analysis/eda/figures/presentation/08_beck_flag_by_department.png)

![Higher-score-group rate across ordinal phone and family responses](../analysis/eda/figures/presentation/09_ordinal_predictor_patterns.png)

![Response patterns across the 22 phone-dependence items](../analysis/eda/figures/presentation/10_tmd_item_response_heatmap.png)

## 4. Cleaning and analysis-table preparation

The preparation step checks ID alignment, missing values, and score consistency. No full rows or student records were removed: there were no duplicate IDs/full rows or missing values in the selected fields, and valid high values were retained. The phone-dependence total is recalculated from its 22 answers because one supplied total was inconsistent. Beck answer items are excluded from model inputs because they define the target. The raw workbooks are not edited.

The next cell reruns the preparation script and loads the derived analysis table.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'analysis' / 'cleaning' / 'prepare_depression_dataset.py')], check=True)
prepared = pd.read_csv(ROOT / 'data' / 'processed' / 'depression_analysis_dataset.csv')
print('Prepared table shape:', prepared.shape)
print('Missing cells in prepared table:', int(prepared.isna().sum().sum()))
display(prepared['Higher_Beck_Score_11plus'].value_counts().rename(index={0: 'Below 11', 1: '11+ (dataset-defined group)'}).to_frame('students'))
display(Markdown((ROOT / 'data' / 'processed' / 'preparation_log.md').read_text()))

## 5. PCA: explore predictor structure

PCA summarizes variation across the input features. It does **not** rank features by how much they affect the Beck target. For this exploration, nominal variables are one-hot encoded, ordered responses are kept in order, inputs are standardized, and Beck score/target/ID are excluded from the PCA calculation. The target is used only to color the PC1–PC2 plot.

The exploratory PCA found 57 encoded dimensions; PC1 and PC2 explain 20.8% of predictor variation together, and 27 components are needed to retain 80%. The groups overlap in the PC1–PC2 view. This full-data PCA is descriptive only; any PCA used by a predictive model must be fit inside training folds.

![PCA scree and cumulative variance](../analysis/pca/figures/01_pca_scree_cumulative_variance.png)

![Students projected onto the first two principal components](../analysis/pca/figures/02_pca_pc1_pc2_by_beck_group.png)

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'analysis' / 'pca' / 'run_pca_eda.py')], check=True)
display(pd.read_csv(ROOT / 'analysis' / 'pca' / 'tables' / 'pca_explained_variance.csv').head(10).round(3))
display(pd.read_csv(ROOT / 'analysis' / 'pca' / 'tables' / 'pca_loadings_top10.csv').assign(max_abs_loading=lambda x: x.iloc[:, 1:].abs().max(axis=1)).sort_values('max_abs_loading', ascending=False).head(15).round(3))

## 6. Modeling plan: stratified 80/20 split

The 80% portion is the **training set**, not a cleaning set. It is used for repeated stratified cross-validation and fitting. The held-out 20% is reserved for final evaluation. The modeling script saves these exact portions as `data/splits/depression_train_80pct.csv` and `data/splits/depression_test_20pct.csv`. To create only those CSVs without refitting models, run `python analysis/modeling/run_depression_models.py --split-only`. A most-common baseline is compared with logistic regression, a decision tree, random forest, and gradient boosting. Each classifier is compared with and without PCA.

The project uses PR-AUC as the main model-selection metric because the 11+ group is smaller. Accuracy alone can be misleading. Precision, recall, F1, balanced accuracy, ROC-AUC, and PR-AUC are reported. A fixed 0.50 probability threshold is used for threshold-based test metrics; it is not a diagnostic threshold.

In [ ]:
# If needed, uncomment and run this once to install the modeling dependencies.
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'analysis' / 'modeling' / 'requirements-modeling.txt')], check=True)

In [ ]:
# This cell runs the 80/20 modeling comparison and writes results/ and models/.
subprocess.run([sys.executable, str(ROOT / 'analysis' / 'modeling' / 'run_depression_models.py')], check=True)
model_results = pd.read_csv(ROOT / 'results' / 'depression_model_comparison.csv')
display(model_results.sort_values('cv_pr_auc_mean', ascending=False).round(3))

### Which inputs help prediction?

After the models are fitted, permutation importance estimates how much the selected model's PR-AUC drops when one original input is shuffled. This answers which inputs the model relies on for prediction in this dataset. It does not prove that an input causes a higher score. A PCA loading is not a substitute for this target-focused analysis.

In [ ]:
importance = pd.read_csv(ROOT / 'results' / 'best_model_permutation_importance.csv')
display(importance.head(15).round(4))
display(Markdown((ROOT / 'results' / 'modeling_summary.md').read_text()))

## 7. Interpretation and limitations

The outcome is a codebook-defined Beck score group, not a clinical diagnosis. The work is based on questionnaire responses from one dataset and cannot establish causation. The scoring guide and questionnaire version should be confirmed with the lecturer. Model performance should be presented with the baseline and class-sensitive measures, and the held-out results should not be described as clinical screening accuracy.

For full details, see `analysis/eda/`, `analysis/cleaning/`, `analysis/pca/`, `analysis/modeling/`, and the generated `results/` files.